# 04 · LoRA with PEFT on a real chat model

Fine-tune a real (tiny) chat LLM — `HuggingFaceTB/SmolLM2-135M-Instruct` — with Hugging Face **PEFT**: pick
`target_modules` from the module names, attach LoRA, teach a new behaviour from 24 examples in a few CPU steps, then
save the adapter, reload it, merge it, and switch between two adapters on one base.

**Time:** ~30 min · **Runs:** offline on CPU in ~1–2 min (uses the cached 270 MB SmolLM2 model) · **Needs:**
[03 · LoRA from scratch](03_lora_from_scratch.ipynb), [02 · SFT with loss masking](02_sft_from_scratch_with_loss_masking.ipynb)

## Why this matters in interviews

- "Walk me through a LoRA fine-tune you ran" — config, target modules, trainable %, LR, what the adapter file is,
  how you served it. This notebook is that walk-through, end to end, with numbers you produced.
- "Which modules go in `target_modules`, and how many weights will you train?" (`tr45`) — answered from the model's
  own module names, with the count checked against PEFT.
- Serving questions — merge vs keep separate, many adapters on one base (`tn34`) — are shown, not just described.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn04` | What is LoRA and why does it work? |
| `tr45` | You are fine-tuning this model with LoRA. Which modules go in target_modules, and how many weights will you train? |
| `tn14` | What hyperparameters matter for LoRA, and how do you set them? |
| `tn34` | How do you serve many fine-tuned models without paying for many GPUs? |
| `tn26` | What is the difference between prompt tuning, prefix tuning and LoRA? (mergeability) |
| `tn20` | How do you evaluate a fine-tuned model before shipping it? |
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `tn24` | What is a chat template and why does it matter when self-hosting? |

In [ ]:
import json, os, tempfile, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
t_start = time.time()
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel, get_peft_model

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
try:
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)   # fp32: CPU training
except Exception as e:
    raise RuntimeError(f"Could not load {MODEL_ID} (needs the HF cache or a one-time 270 MB download).") from e
base.eval()
n_base = sum(p.numel() for p in base.parameters())
print(f"loaded {MODEL_ID}: {n_base:,} parameters, dtype {base.dtype}, in {time.time() - t_start:.1f}s")

## 1. Choose `target_modules` from the module names

**In plain English:** LoRA attaches to named Linear layers. The names you pass to PEFT are literally the names in
`print(model)`. So look before you choose.

In [ ]:
from collections import Counter
linear_names = Counter(name.split(".")[-1] for name, m in base.named_modules() if isinstance(m, torch.nn.Linear))
print("Linear layers by name:", dict(linear_names))
layer0 = base.model.layers[0]
for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    print(f"  self_attn.{name:9s} in={getattr(layer0.self_attn, name).in_features:5d} out={getattr(layer0.self_attn, name).out_features:5d}")
for name in ["gate_proj", "up_proj", "down_proj"]:
    print(f"  mlp.{name:14s} in={getattr(layer0.mlp, name).in_features:5d} out={getattr(layer0.mlp, name).out_features:5d}")
cfg = base.config
print(f"layers={cfg.num_hidden_layers}, hidden={cfg.hidden_size}, heads={cfg.num_attention_heads}, "
      f"kv_heads={cfg.num_key_value_heads} (GQA), tied embeddings={cfg.tie_word_embeddings}")

`k_proj` and `v_proj` are narrower (576 → 192) because SmolLM2 uses grouped-query attention: 9 query heads share 3
key/value heads. `lm_head` shares its weight with `embed_tokens` (tied), so adapting it would also change the input
embeddings. The LoRA parameter count per module is `r · (in + out)`:

In [ ]:
def lora_params(modules, r):
    shapes = {n: (getattr(layer0.self_attn, n) if hasattr(layer0.self_attn, n) else getattr(layer0.mlp, n)) for n in modules}
    return cfg.num_hidden_layers * sum(r * (m.in_features + m.out_features) for m in shapes.values())

options = {"q_proj + v_proj (original LoRA paper)": ["q_proj", "v_proj"],
           "all attention (q, k, v, o)": ["q_proj", "k_proj", "v_proj", "o_proj"],
           "all linear (attention + MLP)": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}
for label, mods in options.items():
    n = lora_params(mods, r=8)
    print(f"r=8, {label:38s}: {n:>9,} trainable ({n / n_base:.3%} of the model)")

We use the configuration from the original LoRA paper — small and fast on a CPU. For real quality you would usually
target all linear layers (`target_modules="all-linear"` in PEFT) and pay ~5× the adapter size.

## 2. Attach LoRA with `LoraConfig` + `get_peft_model`

In [ ]:
lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(base, lora_cfg, adapter_name="json")
model.print_trainable_parameters()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_trainable == lora_params(["q_proj", "v_proj"], 8)          # PEFT agrees with r * (in + out) * layers
assert n_trainable / n_base < 0.01                                   # < 1% of the weights train
print(model.base_model.model.model.layers[0].self_attn.q_proj)

The printout shows what PEFT did: `q_proj` is now a `lora.Linear` holding the original `base_layer` (frozen) plus
`lora_A` (576 → 8) and `lora_B` (8 → 576), with dropout on the adapter path. Same structure as the `LoRALinear` you
wrote in notebook 03.

## 3. The task: always answer in JSON

**A behaviour, not facts.** We teach the model to reply to any question with a JSON object `{"answer": "..."}` —
a format change, the kind of thing fine-tuning is good at (notebook 01). 24 training questions, 6 held-out ones on
*different* topics. Before training, SmolLM2 answers in prose:

In [ ]:
SYSTEM = "You are a helpful assistant."
train_qa = [
    ("What is the capital of France?", "Paris"), ("What is 2 + 2?", "4"), ("What color is the sky on a clear day?", "Blue"),
    ("How many legs does a spider have?", "8"), ("What is the largest planet?", "Jupiter"),
    ("Who wrote Romeo and Juliet?", "William Shakespeare"), ("What is the boiling point of water in Celsius?", "100 degrees"),
    ("What language is spoken in Spain?", "Spanish"), ("What is the chemical symbol for gold?", "Au"),
    ("How many days are in a week?", "7"), ("What is the capital of Japan?", "Tokyo"), ("What animal says moo?", "A cow"),
    ("What is the opposite of hot?", "Cold"), ("What is 10 minus 3?", "7"), ("Which planet is called the red planet?", "Mars"),
    ("What do bees make?", "Honey"), ("What is frozen water called?", "Ice"), ("How many continents are there?", "7"),
    ("What is the capital of Italy?", "Rome"), ("What gas do plants absorb?", "Carbon dioxide"),
    ("What is the fastest land animal?", "The cheetah"), ("What is 5 times 6?", "30"),
    ("What is the capital of Germany?", "Berlin"), ("Which ocean is the largest?", "The Pacific Ocean"),
]
heldout_q = ["What is the capital of Spain?", "How many wheels does a bicycle have?", "What color is grass?",
             "Who painted the Mona Lisa?", "What is 3 plus 5?", "What do cows drink?"]

tok.padding_side = "left"                     # left-pad for batched generation, so every prompt ends at the same place

@torch.no_grad()
def generate(m, questions, max_new_tokens=16):
    msgs = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}] for q in questions]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True, padding=True)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

def is_json_answer(text):
    try:
        return isinstance(json.loads(text), dict) and "answer" in json.loads(text)
    except ValueError:
        return False

t0 = time.time()
with model.disable_adapter():                 # adapter off = the untouched base model
    before = generate(model, heldout_q)
print(f"generated {len(before)} answers in {time.time() - t0:.1f}s")
for q, a in zip(heldout_q, before):
    print(f"{q:38s} -> {a!r}")
json_before = np.mean([is_json_answer(a) for a in before])
print(f"valid JSON answers before training: {json_before:.0%}")

### Build training examples: chat template + loss mask

The same recipe as notebook 02, with the model's real template: tokenize the prompt with
`add_generation_prompt=True`, append the answer and the `<|im_end|>` EOS, and put `-100` on the prompt positions.
Padding is masked **by position**, because SmolLM2's pad token *is* its EOS token.

In [ ]:
def build(question, answer):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True)["input_ids"]
    target = json.dumps({"answer": answer})
    full = tok.apply_chat_template(msgs + [{"role": "assistant", "content": target}], tokenize=True, return_dict=True)["input_ids"]
    assert full[:len(prompt)] == prompt
    full = full[: full.index(tok.eos_token_id, len(prompt)) + 1]           # keep through <|im_end|> (drop the trailing \n)
    return full, [-100] * len(prompt) + full[len(prompt):]

def collate(rows):
    T = max(len(ids) for ids, _ in rows)
    ids = torch.full((len(rows), T), tok.pad_token_id)
    labels = torch.full((len(rows), T), -100)
    attn = torch.zeros((len(rows), T), dtype=torch.long)
    for i, (x, y) in enumerate(rows):                                       # right-pad for training
        ids[i, :len(x)], labels[i, :len(y)], attn[i, :len(x)] = torch.tensor(x), torch.tensor(y), 1
    return {"input_ids": ids, "attention_mask": attn, "labels": labels}

train_rows = [build(q, a) for q, a in train_qa]
ids, labels = train_rows[0]
print("tokens in example 0:", len(ids), "| learned:", sum(l != -100 for l in labels))
print("learned text:", repr(tok.decode([l for l in labels if l != -100])))
assert labels[-1] == tok.eos_token_id                                       # EOS is inside the loss

## 4. Train: a short manual loop

A plain PyTorch loop — the same four lines as any training loop. `model(**batch)` returns the loss because we pass
`labels`; Hugging Face shifts them and ignores `-100` internally. LoRA learning rates are ~10× a full fine-tune
(`tn14`); on a 24-example toy we go higher still to finish in a few steps.

In [ ]:
val_rows = [build(q, a) for q, a in [("What is the capital of Spain?", "Madrid"), ("What color is grass?", "Green"),
                                     ("What is 3 plus 5?", "8"), ("How many wheels does a bicycle have?", "2")]]
val_batch = collate(val_rows)

def val_loss(m):
    m.eval()
    with torch.no_grad():
        return m(**val_batch).loss.item()

def train_adapter(m, rows, steps, lr, batch_size=4, seed=0, val_every=3):
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    g = torch.Generator().manual_seed(seed)
    history = []
    for step in range(1, steps + 1):
        m.train()
        idx = torch.randperm(len(rows), generator=g)[:batch_size].tolist()
        loss = m(**collate([rows[i] for i in idx])).loss
        loss.backward(); opt.step(); opt.zero_grad()
        history.append({"step": step, "train_loss": loss.item(), "val_loss": val_loss(m) if val_every and step % val_every == 0 else None})
    m.eval()
    return history

val_before = val_loss(model)
print(f"val loss before: {val_before:.3f}")
t0 = time.time()
hist = train_adapter(model, train_rows, steps=10, lr=2e-3)
print(f"10 steps (batch 4) in {time.time() - t0:.1f}s")
for h in hist:
    print(f"step {h['step']:2d}  train loss {h['train_loss']:.3f}" + (f"   val loss {h['val_loss']:.3f}" if h["val_loss"] else ""))

In [ ]:
plt.figure(figsize=(7, 3.4))
plt.plot([h["step"] for h in hist], [h["train_loss"] for h in hist], "o-", label="train (batch of 4)")
vs = [(h["step"], h["val_loss"]) for h in hist if h["val_loss"]]
plt.plot([0] + [s for s, _ in vs], [val_before] + [v for _, v in vs], "s--", label="held-out (4 questions)")
plt.xlabel("step"); plt.ylabel("loss on answer tokens"); plt.yscale("log"); plt.legend()
plt.title("LoRA on SmolLM2-135M: 24 examples, 10 steps"); plt.show()

## 5. Before / after on held-out questions

In [ ]:
t0 = time.time()
after = generate(model, heldout_q)
print(f"generated in {time.time() - t0:.1f}s\n")
for q, b, a in zip(heldout_q, before, after):
    print(f"Q: {q}\n   before: {b!r}\n   after : {a!r}")
json_after = np.mean([is_json_answer(a) for a in after])
print(f"\nvalid JSON answers: before {json_before:.0%} -> after {json_after:.0%}")
assert json_after >= 0.8 > json_before

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.bar(["base model", "base + LoRA adapter"], [json_before, json_after], color=["#adb5bd", "#2f9e44"])
plt.ylim(0, 1.05); plt.ylabel("held-out answers that parse as JSON")
plt.title("The behaviour transfers to unseen questions"); plt.show()

The *format* generalised to questions the adapter never saw — that is what 24 examples can teach. Check the
*content* too: some answers may be wrong (a 135M model knows little), and a format adapter does not make it know more.
Evaluating a fine-tune means both: the behaviour you trained (JSON rate) **and** the things you did not (accuracy,
refusals, general ability — `tn20`). With this few examples it is also easy to over-train: when the held-out loss
stops falling while the training loss keeps dropping, stop (`tn28`).

## 6. Save the adapter — and compare its size to the base

In [ ]:
work = Path(tempfile.mkdtemp(prefix="lora04_"))
model.save_pretrained(work / "json_adapter")
files = {f.name: f.stat().st_size for f in (work / "json_adapter" / "json").iterdir()} if (work / "json_adapter" / "json").exists() \
    else {f.name: f.stat().st_size for f in (work / "json_adapter").iterdir()}
print("adapter files:", {k: f"{v / 1024:.1f} KB" for k, v in files.items()})

from huggingface_hub import try_to_load_from_cache
base_file = try_to_load_from_cache(MODEL_ID, "model.safetensors")
base_mb = os.path.getsize(base_file) / 1e6 if isinstance(base_file, str) else n_base * 2 / 1e6
adapter_mb = files["adapter_model.safetensors"] / 1e6
print(f"adapter: {adapter_mb:.2f} MB   base checkpoint (bf16 on disk): {base_mb:.0f} MB   ratio 1 : {base_mb / adapter_mb:.0f}")

plt.figure(figsize=(6, 3.2))
plt.bar(["base model", "LoRA adapter"], [base_mb, adapter_mb], color=["#adb5bd", "#2f9e44"])
plt.yscale("log"); plt.ylabel("MB on disk (log)"); plt.title("You ship the adapter, not the model"); plt.show()

`save_pretrained` on a `PeftModel` writes only the adapter (`adapter_model.safetensors`) and its
`adapter_config.json` — which records the base model's name, `r`, `lora_alpha` and `target_modules`. (The adapter is
saved in fp32 here; a bf16 adapter would be half the size.) For a 7B model with `all-linear` at r = 16 the adapter is
~80 MB in bf16 against a 13 GB base.

## 7. Reload with `PeftModel.from_pretrained`, then merge

Reloading needs a fresh copy of the base (the old one now has LoRA layers injected into it).

In [ ]:
adapter_dir = work / "json_adapter" / "json" if (work / "json_adapter" / "json").exists() else work / "json_adapter"
fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
reloaded = PeftModel.from_pretrained(fresh_base, adapter_dir, adapter_name="json").eval()

probe = collate(val_rows)
with torch.no_grad():
    logits_trained = model(**probe).logits
    logits_reloaded = reloaded(**probe).logits
print(f"max |trained - reloaded| logit: {(logits_trained - logits_reloaded).abs().max():.2e}")
assert torch.allclose(logits_trained, logits_reloaded, atol=1e-4)

merged = reloaded.merge_and_unload()            # folds (alpha/r) * B @ A into each W, returns a plain transformers model
with torch.no_grad():
    logits_merged = merged(**probe).logits
print(f"max |adapter - merged| logit: {(logits_trained - logits_merged).abs().max():.2e}   type now: {type(merged).__name__}")
assert torch.allclose(logits_trained, logits_merged, atol=1e-3)
assert not any("lora" in n for n, _ in merged.named_modules())
print("merged model generates:", generate(merged, heldout_q[:2]))

The merged model is an ordinary `LlamaForCausalLM` with the original parameter count: serve it with any inference
stack and there is no LoRA overhead. The tiny logit differences are float rounding from adding `B @ A` into `W`.

## 8. Two adapters, one base: `set_adapter`

Keep adapters **unmerged** when you want one base in memory and many behaviours. Train a second adapter — "answer
in capitals, one line" — on the *same* `PeftModel`, then switch.

In [ ]:
caps_rows = []
for q, a in train_qa:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True)["input_ids"]
    full = tok.apply_chat_template(msgs + [{"role": "assistant", "content": f"ANSWER: {a.upper()}."}],
                                   tokenize=True, return_dict=True)["input_ids"]
    full = full[: full.index(tok.eos_token_id, len(prompt)) + 1]
    caps_rows.append((full, [-100] * len(prompt) + full[len(prompt):]))

model.add_adapter("caps", lora_cfg)
model.set_adapter("caps")                         # only the active adapter is trainable
print("active:", model.active_adapter, "| trainable now:", sum(p.numel() for p in model.parameters() if p.requires_grad))
t0 = time.time()
caps_hist = train_adapter(model, caps_rows, steps=6, lr=3e-3, seed=1, val_every=0)
print(f"trained 'caps' in {time.time() - t0:.1f}s, last loss {caps_hist[-1]['train_loss']:.3f}")

In [ ]:
outputs = {}
for name in ["json", "caps"]:
    model.set_adapter(name)
    outputs[name] = generate(model, heldout_q[:3])
with model.disable_adapter():
    outputs["base (disabled)"] = generate(model, heldout_q[:3])
for i, q in enumerate(heldout_q[:3]):
    print(f"Q: {q}")
    for name, outs in outputs.items():
        print(f"   {name:15s}: {outs[i]!r}")
assert all(is_json_answer(o) for o in outputs["json"]) and all(o.startswith("ANSWER") for o in outputs["caps"])
print("\nadapters on this base:", list(model.peft_config))

One base model in memory, two behaviours, switched per call with `set_adapter` (or bypassed with
`disable_adapter()`). Serving stacks take this further: vLLM and S-LoRA batch requests that use *different* adapters
in the same forward pass, which is what makes per-customer fine-tunes affordable (`tn34`).

| Serving choice | Pros | Cons |
|---|---|---|
| merge (`merge_and_unload`) | zero overhead; any inference server | one model copy per behaviour |
| keep adapters separate | one base, many small adapters, hot-swap | small latency overhead; server must support LoRA |

## Try it yourself

**1.** Use `LoraConfig(target_modules="all-linear", r=8)` on a fresh base and print the trainable parameters. Check
the number against `lora_params` for all seven projections.

In [ ]:
# Solution — try it yourself first, then run this
probe_base = merged          # the merged model from section 7 is a plain LlamaForCausalLM again - reuse it
all_lin = get_peft_model(probe_base, LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM"))
all_lin.print_trainable_parameters()
n_all = sum(p.numel() for p in all_lin.parameters() if p.requires_grad)
print("formula:", f"{lora_params(options['all linear (attention + MLP)'], 8):,}", "| PEFT:", f"{n_all:,}")
print("lm_head wrapped?", any(n.endswith("lm_head.lora_A.default.weight") for n, _ in all_lin.named_parameters()))

PEFT's `"all-linear"` wraps every Linear layer **except the output head** — good, because here `lm_head` is tied to
the input embeddings.

**2.** How big would the `json` adapter be for a 7B model (32 layers, hidden 4096) with `q_proj` + `v_proj` at r = 8,
stored in bf16? And with all linear layers (MLP 11008) at r = 16?

In [ ]:
# Solution — try it yourself first, then run this
qv_7b = 32 * 8 * ((4096 + 4096) + (4096 + 4096))
all_7b = 32 * 16 * (4 * (4096 + 4096) + 2 * (4096 + 11008) + (11008 + 4096))
print(f"q+v r=8   : {qv_7b:,} params -> {qv_7b * 2 / 1e6:.1f} MB in bf16")
print(f"all r=16  : {all_7b:,} params -> {all_7b * 2 / 1e6:.1f} MB in bf16 (vs ~13,500 MB for the base)")

**3.** Read the saved `adapter_config.json`. Which fields would a serving system need to load it correctly?

In [ ]:
# Solution — try it yourself first, then run this
conf = json.loads((adapter_dir / "adapter_config.json").read_text())
print({k: conf[k] for k in ["base_model_name_or_path", "peft_type", "task_type", "r", "lora_alpha", "target_modules"]})
print(f"notebook total runtime so far: {time.time() - t_start:.0f}s")

## Say it in an interview

- **The recipe:** `LoraConfig(r, lora_alpha, lora_dropout, target_modules, task_type="CAUSAL_LM")` →
  `get_peft_model` → `print_trainable_parameters()` (here 0.34%) → an ordinary training loop on chat-templated,
  loss-masked examples → `save_pretrained` (a ~2 MB adapter plus its config) → `PeftModel.from_pretrained` on the
  base to reload → `merge_and_unload()` for a plain model with zero inference overhead.
- **Target modules:** read them from `print(model)`; the count is `layers × Σ r·(in + out)`. q + v is the minimum;
  `all-linear` (attention + MLP) usually wins on quality. Watch tied `lm_head`/embeddings.
- **What 24 examples did:** changed the *format* on unseen questions (JSON rate before vs after), not the model's
  knowledge — measure both.
- **Serving:** merge for one behaviour; keep adapters separate and `set_adapter` / multi-LoRA batching for many
  tenants on one base.
- **Traps:** pad == eos masking away the EOS; a different chat template or system prompt at inference than in
  training; full-fine-tune learning rates (nothing moves); forgetting to evaluate what you did *not* train.

## Next

- [05 · QLoRA with bitsandbytes and PEFT](05_qlora_with_bitsandbytes_and_peft.ipynb) — the same adapter on a 4-bit base
- [06 · DPO preference tuning](06_dpo_preference_tuning.ipynb) — beyond SFT: learning from chosen vs rejected
- [Trainer API: fine-tune a small model](../10_huggingface/05_trainer_api_fine_tune_small_model.ipynb) · [Tokenizers, models and generate](../10_huggingface/02_tokenizers_models_and_generate.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)